# 01 · Merge the agentic-extraction LoRA → merged HF checkpoint (Unsloth) → push to HF

Merges the published adapter `Koalacrown/qwen3.6-27b-agentic-extraction-sft-lora` into its stock base
(auto-resolved from the adapter config), producing a **single merged bf16 HF checkpoint** and **pushing
it to the Hub**. Starts from the *published* adapter, so no Tinker key is needed.

**Why merge?** A merged checkpoint loads as an ordinary HF model, so:
- downstream activation/probe notebooks can read its residual-stream activations by repo id (needs raw HF weights, not a vLLM endpoint);
- it sidesteps strict `--lora-modules` loaders that can reject a raw adapter over output-embedding LoRA-naming wrinkles.

Output lives on HF (`MERGED_REPO`), so any later runtime just pulls it — no Drive copy.

### Runtime
- **GPU: A100 80GB or H100** — this is a **27B** base loaded in 16-bit for a clean merge (~54GB weights). An L4/24GB or A100-40GB will OOM.
- **Disk:** ~55GB local scratch for the merge before upload → keep ~70GB free.

## 1. Install Unsloth

In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 2. Config + HF login
Paste an HF token with **write** access (Settings → Access Tokens), or store it as a Colab secret
named `HF_TOKEN`. `MERGED_REPO` is the destination — downstream notebooks and any vLLM server pull
the merged checkpoint straight from there.

In [ ]:
import os
ADAPTER_REPO = "Koalacrown/qwen3.6-27b-agentic-extraction-sft-lora"   # published agentic-extraction SFT adapter (PEFT)
MERGED_REPO  = "Koalacrown/qwen3.6-27b-agentic-extraction-sft-merged" # destination: merged bf16 HF checkpoint
MERGED_LOCAL = "/content/agentic_merged"                              # local scratch staged before upload
MAX_SEQ_LEN  = 40960  # native context (only affects load + the sanity gen)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata; HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception: pass
if not HF_TOKEN:
    from getpass import getpass; HF_TOKEN = getpass("HF write token: ")
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login; login(token=HF_TOKEN)
print("merge", ADAPTER_REPO, "-> (base from adapter config), push ->", MERGED_REPO)

## 3. Load adapter (auto-pulls base) in 16-bit
`from_pretrained` reads `base_model_name_or_path` from the adapter config, downloads the base model,
and attaches the LoRA. `load_in_4bit=False` → a clean full-precision merge.

> If this step **warns** about an unexpected `unembed_tokens` / `lm_head` adapter key, that's an
> output-embedding LoRA; Unsloth merges the rest faithfully (the trained behaviour lives in attn/MLP).

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = ADAPTER_REPO,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,        # auto: bf16 on Ampere+
    load_in_4bit   = False,       # full precision so the merge is faithful
    token          = os.environ.get("HF_TOKEN"),
)
print("loaded:", type(model).__name__)

## 4. Sanity — confirm the merged adapter behaves
Run a quick generation to confirm the merged behaviour looks right before saving. `enable_thinking`
is left OFF; flip it if this adapter was trained with thinking on.

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [{"role": "user", "content": "Extract the key entities and any action items from this note: 'Met with Dana on Tuesday; she will send the Q3 budget by Friday and wants a follow-up call next week.'"}]
prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=300, temperature=0.7, top_p=0.95)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## 5. Merge → push merged bf16 HF checkpoint to the Hub
`save_method="merged_16bit"` merges the LoRA into ordinary HF weights (config.json + safetensors +
tokenizer) and uploads them to `MERGED_REPO` in one call. ~16GB upload.

In [ ]:
model.push_to_hub_merged(MERGED_REPO, tokenizer, save_method="merged_16bit", token=os.environ["HF_TOKEN"])
print("pushed:", f"https://huggingface.co/{MERGED_REPO}")

### Fallback: stage locally, then upload manually
Only if the push above fails on the upload step (flaky link) — the merge is the slow part, so
re-upload the staged folder without re-merging.

In [ ]:
# Only run this if the push above failed.
# model.save_pretrained_merged(MERGED_LOCAL, tokenizer, save_method="merged_16bit")
# from huggingface_hub import HfApi
# api = HfApi()
# api.create_repo(MERGED_REPO, repo_type="model", private=False, exist_ok=True, token=os.environ["HF_TOKEN"])
# api.upload_folder(folder_path=MERGED_LOCAL, repo_id=MERGED_REPO, token=os.environ["HF_TOKEN"])
# print("uploaded", MERGED_LOCAL, "->", MERGED_REPO)

Merged checkpoint on the Hub. Downstream notebooks can now load `MERGED_REPO` by repo id (raw HF
weights for activation/probe work), or a vLLM server can serve it directly as a full model — no
`--lora-modules`, no Drive copy, it just pulls from HF.